# DREAM

In [ ]:
import scipp as sc
import tof

import scippnexus as snx
from ess.reduce.unwrap import GenericUnwrapWorkflow
from ess.reduce.nexus.types import *
from ess.reduce.unwrap.types import *
from ess.reduce.unwrap.lut import LtotalRange, ChopperFrameSequence

source = tof.Source(facility="ess", neutrons=1_000_000, pulses=2)

## High-flux mode

### Chopper parameters

In [ ]:
choppers = {
    "psc1": {
        "frequency": {"value": 210.0, "unit": "Hz"},
        "open": {
            "value": [-1.23, 70.49, 84.765, 113.565, 170.29, 271.635, 286.035, 301.17],
            "unit": "deg",
        },
        "close": {
            "value": [1.23, 73.51, 88.035, 116.835, 175.31, 275.565, 289.965, 303.63],
            "unit": "deg",
        },
        "distance": {"value": 6.145, "unit": "m"},
        "phase": {"value": -155.0, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "psc1",
    },
    "psc2": {
        "frequency": {"value": 196.0, "unit": "Hz"},
        "open": {
            "value": [-1.23, 27.0, 55.8, 142.385, 156.765, 214.115, 257.23, 315.49],
            "unit": "deg",
        },
        "close": {
            "value": [1.23, 30.6, 59.4, 145.615, 160.035, 217.885, 261.17, 318.11],
            "unit": "deg",
        },
        "distance": {"value": 6.155, "unit": "m"},
        "phase": {"value": 100.5, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "psc2",
    },
    "oc": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [-13.8], "unit": "deg"},
        "close": {"value": [13.8], "unit": "deg"},
        "distance": {"value": 6.174, "unit": "m"},
        "phase": {"value": 27.0, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "oc",
    },
    "bcc": {
        "frequency": {"value": 112.0, "unit": "Hz"},
        "open": {"value": [-36.875, 143.125], "unit": "deg"},
        "close": {"value": [36.875, 216.875], "unit": "deg"},
        "distance": {"value": 9.78, "unit": "m"},
        "phase": {"value": 20.0, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "bcc",
    },
    "t0": {
        "frequency": {"value": 28.0, "unit": "Hz"},
        "open": {"value": [-157.45], "unit": "deg"},
        "close": {"value": [157.45], "unit": "deg"},
        "distance": {"value": 13.05, "unit": "m"},
        "phase": {"value": 90.0, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "t0",
    },
}

In [ ]:
dream_choppers = {}
for key, ch in choppers.items():
    dream_choppers[key] = tof.Chopper.from_json(name=key, params=ch).to_diskchopper()

for key, ch in dream_choppers.items():
    print(key)
    display(ch)

### Tof model

In [ ]:
source = tof.Source(facility="ess", neutrons=1_000_000, pulses=2)
source_position = sc.vector([0, 0, 0], unit="m")
detector = tof.Detector(distance=sc.scalar(77.0, unit="m"), name="detector")

params = [
    tof.Chopper.from_diskchopper(ch, name=key) for key, ch in dream_choppers.items()
] + [detector]

model = tof.Model(source=source, components=params)
res = model.run()
res.plot()

In [ ]:
res["detector"].plot()

### Wavelength lookup table

In [ ]:
wf = GenericUnwrapWorkflow(
    run_types=[SampleRun], monitor_types=[], wavelength_from="analytical"
)

wf[DiskChoppers[SampleRun]] = dream_choppers
wf[LtotalRange[SampleRun, snx.NXdetector]] = sc.scalar(5, unit="m"), detector.distance
wf[Position[snx.NXsource, SampleRun]] = source_position

table = wf.compute(LookupTable[SampleRun, snx.NXdetector])
table.plot()

In [ ]:
frames = wf.compute(ChopperFrameSequence[SampleRun])
at_sample = frames.propagate_to(detector.distance)
at_sample.draw()

## High-resolution mode

### Chopper parameters

In [ ]:
choppers = {
    "psc1": {
        "frequency": {"value": 210.0, "unit": "Hz"},
        "open": {
            "value": [-1.23, 70.49, 84.765, 113.565, 170.29, 271.635, 286.035, 301.17],
            "unit": "deg",
        },
        "close": {
            "value": [1.23, 73.51, 88.035, 116.835, 175.31, 275.565, 289.965, 303.63],
            "unit": "deg",
        },
        "distance": {"value": 6.145, "unit": "m"},
        "phase": {"value": -155.0, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "psc1",
    },
    "psc2": {
        "frequency": {"value": 196.0, "unit": "Hz"},
        "open": {
            "value": [-1.23, 27.0, 55.8, 142.385, 156.765, 214.115, 257.23, 315.49],
            "unit": "deg",
        },
        "close": {
            "value": [1.23, 30.6, 59.4, 145.615, 160.035, 217.885, 261.17, 318.11],
            "unit": "deg",
        },
        "distance": {"value": 6.155, "unit": "m"},
        "phase": {"value": 100.5, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "psc2",
    },
    "oc": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [-13.8], "unit": "deg"},
        "close": {"value": [13.8], "unit": "deg"},
        "distance": {"value": 6.174, "unit": "m"},
        "phase": {"value": 27.0, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "oc",
    },
    "bcc": {
        "frequency": {"value": 112.0, "unit": "Hz"},
        "open": {"value": [-36.875, 143.125], "unit": "deg"},
        "close": {"value": [36.875, 216.875], "unit": "deg"},
        "distance": {"value": 9.78, "unit": "m"},
        "phase": {"value": 20.0, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "bcc",
    },
    "t0": {
        "frequency": {"value": 28.0, "unit": "Hz"},
        "open": {"value": [-157.45], "unit": "deg"},
        "close": {"value": [157.45], "unit": "deg"},
        "distance": {"value": 13.05, "unit": "m"},
        "phase": {"value": 90.0, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "t0",
    },
}

In [ ]:
dream_choppers = {}
for key, ch in choppers.items():
    dream_choppers[key] = tof.Chopper.from_json(name=key, params=ch).to_diskchopper()

for key, ch in dream_choppers.items():
    print(key)
    display(ch)

### Tof model

In [ ]:
params = [
    tof.Chopper.from_diskchopper(ch, name=key) for key, ch in dream_choppers.items()
] + [detector]

model = tof.Model(source=source, components=params)
res = model.run()
res.plot()

In [ ]:
res["detector"].plot()

### Wavelength lookup table

In [ ]:
wf = GenericUnwrapWorkflow(
    run_types=[SampleRun], monitor_types=[], wavelength_from="analytical"
)

wf[DiskChoppers[SampleRun]] = dream_choppers
wf[LtotalRange[SampleRun, snx.NXdetector]] = sc.scalar(5, unit="m"), detector.distance
wf[Position[snx.NXsource, SampleRun]] = source_position

table = wf.compute(LookupTable[SampleRun, snx.NXdetector])
table.plot()

In [ ]:
frames = wf.compute(ChopperFrameSequence[SampleRun])
at_sample = frames.propagate_to(detector.distance)
at_sample.draw()